In [5]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

from pathlib import Path
import dill

with open(f"{Path.home()}/tutorial_5_best_model.pkl", "rb") as f:
    base_model = dill.load(f)


    
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


# Task 2
In Section 1 of Tutorial 6, when defining the search space, a number of layers are imported, however only LinearInteger and the full precision nn.Linear are selected.

-    Now, extend the search to consider all supported precisions for the Linear layer in Mase, including Minifloat, BlockFP, BlockLog, Binary, etc. This may also require changing the model constructor so the required arguments are passed when instantiating each layer.

-    Run the search again, and plot a figure that has the number of trials on the x axis, and the maximum achieved accuracy up to that point on the y axis. Plot one curve for each precision to compare their performance.



In [6]:
# Defining Search Space

import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    LinearBinaryResidualSign,
)

search_space = {
    "frac_width": [2, 4, 8],
    "int_width": [8, 16, 32],
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger,
        LinearMinifloatDenorm,
        LinearMinifloatIEEE,
        LinearLog,
        LinearBlockFP,
        LinearBlockMinifloat,
        LinearBlockLog,
        LinearBinary,
        LinearBinaryScaling,
    ],
}


from chop.tools.utils import deepsetattr
from copy import deepcopy


def construct_model(trial):

    # Fetch the model
    trial_model = deepcopy(base_model)

    # Quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == torch.nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

# If the chosen layer is integer, define the low precision config
            if new_layer_cls == LinearInteger:
                kwargs["config"] = {
                    "data_in_width": 8,
                    "data_in_frac_width": 4,
                    "weight_width": 8,
                    "weight_frac_width": 4,
                    "bias_width": 8,
                    "bias_frac_width": 4,
                }
            elif new_layer_cls == LinearMinifloatDenorm:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_width": 2,
                    "weight_exponent_bias": 1,
                    "data_in_width": 8,
                    "data_in_exponent_width": 2,
                    "data_in_exponent_bias": 1,
                    "bias_width": 8,
                    "bias_exponent_width": 2,
                    "bias_exponent_bias": 1,
                }
            elif new_layer_cls == LinearMinifloatIEEE:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_width": 3,
                    "weight_exponent_bias": 3,
                    "data_in_width": 8,
                    "data_in_exponent_width": 3,
                    "data_in_exponent_bias": 3,
                    "bias_width": 8,
                    "bias_exponent_width": 3,
                    "bias_exponent_bias": 3,
                }
            elif new_layer_cls == LinearLog:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_bias": 127,
                    "data_in_width": 8,
                    "data_in_exponent_bias": 127,
                    "bias_width": 8,
                    "bias_exponent_bias": 127,
                }
            elif new_layer_cls == LinearBlockFP:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_width": 8,
                    "weight_exponent_bias": 127,
                    "weight_block_size": 32,
                    "data_in_width": 8,
                    "data_in_exponent_width": 8,
                    "data_in_exponent_bias": 127,
                    "data_in_block_size": 32,
                    "bias_width": 8,
                    "bias_exponent_width": 8,
                    "bias_exponent_bias": 127,
                    "bias_block_size": 32,
                }
            elif new_layer_cls == LinearBlockMinifloat:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_width": 3,
                    "weight_exponent_bias_width": 3,
                    "weight_block_size": 32,
                    "data_in_width": 8,
                    "data_in_exponent_width": 3,
                    "data_in_exponent_bias_width": 3,
                    "data_in_block_size": 32,
                    "bias_width": 8,
                    "bias_exponent_width": 3,
                    "bias_exponent_bias_width": 3,
                    "bias_block_size": 32,
                }
            elif new_layer_cls == LinearBlockLog:
                kwargs["config"] = {
                    "weight_width": 8,
                    "weight_exponent_bias_width": 3,
                    "weight_block_size": 32,
                    "data_in_width": 8,
                    "data_in_exponent_bias_width": 3,
                    "data_in_block_size": 32,
                    "bias_width": 8,
                    "bias_exponent_bias_width": 3,
                    "bias_block_size": 32,
                }
            elif new_layer_cls == LinearBinary:
                kwargs["config"] = {
                    "weight_stochastic": False,
                    "weight_bipolar": True,
                }
            elif new_layer_cls == LinearBinaryScaling:
                kwargs["config"] = {
                    "data_in_stochastic": False,
                    "bias_stochastic": False,
                    "weight_stochastic": False,
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            elif new_layer_cls == LinearBinaryResidualSign:
                # Based on the provided snippet, this raises NotImplementedError
                pass


            # Create the new layer (copy the weights)
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data

            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)

    return trial_model

# Making layer hyperparameters searchable. <GEMINI> make sure to use the following code snippet as a template for how to append hyperparams to the search space AND incorpate this into `construct_model`


In [7]:
# Defining Search Space

import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    LinearBinaryResidualSign,
)

search_space = {
    "frac_width": [2, 4, 8],
    "int_width": [8, 16, 32],
    "exponent_width": [2, 3, 4, 5],
    "block_size": [16, 32, 64],
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger,
        LinearMinifloatDenorm,
        LinearMinifloatIEEE,
        LinearLog,
        LinearBlockFP,
        LinearBlockMinifloat,
        LinearBlockLog,
        LinearBinary,
        LinearBinaryScaling,
    ],
}


from chop.tools.utils import deepsetattr
from copy import deepcopy


from chop.tools.utils import deepsetattr
from copy import deepcopy

def construct_model(trial):
    # Fetch the model
    trial_model = deepcopy(base_model)

    # Quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == torch.nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # --- Integer ---
            if new_layer_cls == LinearInteger:
                idx_w = trial.suggest_int(f"{name}_int_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_f = trial.suggest_int(f"{name}_frac_width_idx", 0, len(search_space["frac_width"]) - 1)
                
                kwargs["config"] = {
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_frac_width": search_space["frac_width"][idx_f],
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_frac_width": search_space["frac_width"][idx_f],
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_frac_width": search_space["frac_width"][idx_f],
                }

            # --- Minifloat Denorm ---
            elif new_layer_cls == LinearMinifloatDenorm:
                idx_w = trial.suggest_int(f"{name}_mf_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_e = trial.suggest_int(f"{name}_mf_exp_idx", 0, len(search_space["exponent_width"]) - 1)
                ew = search_space["exponent_width"][idx_e]
                
                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_width": ew,
                    "weight_exponent_bias": (2**(ew-1))-1,
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_width": ew,
                    "data_in_exponent_bias": (2**(ew-1))-1,
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_width": ew,
                    "bias_exponent_bias": (2**(ew-1))-1,
                }

            # --- Minifloat IEEE ---
            elif new_layer_cls == LinearMinifloatIEEE:
                idx_w = trial.suggest_int(f"{name}_ieee_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_e = trial.suggest_int(f"{name}_ieee_exp_idx", 0, len(search_space["exponent_width"]) - 1)
                ew = search_space["exponent_width"][idx_e]

                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_width": ew,
                    "weight_exponent_bias": (2**(ew-1))-1,
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_width": ew,
                    "data_in_exponent_bias": (2**(ew-1))-1,
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_width": ew,
                    "bias_exponent_bias": (2**(ew-1))-1,
                }

            # --- Log ---
            elif new_layer_cls == LinearLog:
                idx_w = trial.suggest_int(f"{name}_log_width_idx", 0, len(search_space["int_width"]) - 1)
                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_bias": 127,
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_bias": 127,
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_bias": 127,
                }

            # --- Block FP ---
            elif new_layer_cls == LinearBlockFP:
                idx_w = trial.suggest_int(f"{name}_bfp_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_b = trial.suggest_int(f"{name}_bfp_block_idx", 0, len(search_space["block_size"]) - 1)
                bs = search_space["block_size"][idx_b]

                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_width": 8,
                    "weight_exponent_bias": 127,
                    "weight_block_size": [bs],
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_width": 8,
                    "data_in_exponent_bias": 127,
                    "data_in_block_size": [bs],
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_width": 8,
                    "bias_exponent_bias": 127,
                    "bias_block_size": [bs],
                }

            # --- Block Minifloat ---
            elif new_layer_cls == LinearBlockMinifloat:
                idx_w = trial.suggest_int(f"{name}_bmf_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_e = trial.suggest_int(f"{name}_bmf_exp_idx", 0, len(search_space["exponent_width"]) - 1)
                idx_b = trial.suggest_int(f"{name}_bmf_block_idx", 0, len(search_space["block_size"]) - 1)
                
                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_width": search_space["exponent_width"][idx_e],
                    "weight_exponent_bias_width": search_space["exponent_width"][idx_e],
                    "weight_block_size": [search_space["block_size"][idx_b]],
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_width": search_space["exponent_width"][idx_e],
                    "data_in_exponent_bias_width": search_space["exponent_width"][idx_e],
                    "data_in_block_size": [search_space["block_size"][idx_b]],
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_width": search_space["exponent_width"][idx_e],
                    "bias_exponent_bias_width": search_space["exponent_width"][idx_e],
                    "bias_block_size": [search_space["block_size"][idx_b]],
                }

            # --- Block Log ---
            elif new_layer_cls == LinearBlockLog:
                idx_w = trial.suggest_int(f"{name}_blog_width_idx", 0, len(search_space["int_width"]) - 1)
                idx_b = trial.suggest_int(f"{name}_blog_block_idx", 0, len(search_space["block_size"]) - 1)

                kwargs["config"] = {
                    "weight_width": search_space["int_width"][idx_w],
                    "weight_exponent_bias_width": 3,
                    "weight_block_size": [search_space["block_size"][idx_b]],
                    "data_in_width": search_space["int_width"][idx_w],
                    "data_in_exponent_bias_width": 3,
                    "data_in_block_size": [search_space["block_size"][idx_b]],
                    "bias_width": search_space["int_width"][idx_w],
                    "bias_exponent_bias_width": 3,
                    "bias_block_size": [search_space["block_size"][idx_b]],
                }

            # --- Binary ---
            elif new_layer_cls == LinearBinary:
                kwargs["config"] = {
                    "weight_stochastic": trial.suggest_categorical(f"{name}_bin_stoch", [True, False]),
                    "weight_bipolar": trial.suggest_categorical(f"{name}_bin_bipol", [True, False]),
                }

            # --- Binary Scaling ---
            elif new_layer_cls == LinearBinaryScaling:
                kwargs["config"] = {
                    "data_in_stochastic": trial.suggest_categorical(f"{name}_bins_in_stoch", [True, False]),
                    "bias_stochastic": False,
                    "weight_stochastic": trial.suggest_categorical(f"{name}_bins_w_stoch", [True, False]),
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            
            # --- Binary Residual Sign ---
            elif new_layer_cls == LinearBinaryResidualSign:
                # Placeholder for implementation
                pass

            # Create the new layer (copy the weights)
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()

            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)

    return trial_model


# Run Study

In [8]:
from chop.tools import get_trainer
import random


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]


from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
)


[I 2026-02-03 17:01:05,273] A new study created in memory with name: bert-tiny-nas-study
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.quantized.modules.linear.LinearInteger'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.quantized.modules.linear.LinearM

TypeError: BlockMinifloatQuantize.backward() missing 3 required positional arguments: 'width', 'exponent_width', and 'exponent_bias_width'